# Проект: Предсказание риска сердечного приступа

Цель проекта — построить модель машинного обучения для определения риска сердечного приступа на основе медицинских и поведенческих факторов.

In [2]:
import pandas as pd
import numpy as np
import joblib
import os
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

train = pd.read_csv('../data/heart_train.csv')
test = pd.read_csv('../data/heart_test.csv')

display("Train shape:", train.shape)
display("Test shape:", test.shape)

train.head()

'Train shape:'

(8685, 28)

'Test shape:'

(966, 27)

,Unnamed: 0,Age,Cholesterol,Heart rate,Diabetes,Family History,Smoking,Obesity,Alcohol Consumption,Exercise Hours Per Week,...,Physical Activity Days Per Week,Sleep Hours Per Day,Heart Attack Risk (Binary),Blood sugar,CK-MB,Troponin,Gender,Systolic blood pressure,Diastolic blood pressure,id
0,0,0.359551,0.732143,0.074244,1.0,1.0,1.0,1.0,1.0,0.535505,...,3.0,0.333333,0.0,0.227018,0.048229,0.036512,Male,0.212903,0.709302,2664
1,1,0.202247,0.325000,0.047663,1.0,1.0,0.0,0.0,1.0,0.068690,...,3.0,0.833333,0.0,0.150198,0.017616,0.000194,Female,0.412903,0.569767,9287
2,2,0.606742,0.860714,0.055912,1.0,0.0,1.0,1.0,1.0,0.944001,...,2.0,1.000000,0.0,0.227018,0.048229,0.036512,Female,0.238710,0.220930,5379
3,3,0.730337,0.007143,0.053162,0.0,0.0,1.0,0.0,1.0,0.697023,...,0.0,0.333333,1.0,0.227018,0.048229,0.036512,Female,0.348387,0.267442,8222
4,4,0.775281,0.757143,0.021998,0.0,0.0,1.0,0.0,1.0,0.412878,...,5.0,1.000000,1.0,0.227018,0.048229,0.036512,Male,0.619355,0.441860,4047


## Первичный анализ данных

На первом этапе были загружены обучающая и тестовая выборки, после чего изучены их размеры и общая структура.

Выявлено:
- train-набор содержит 8685 строк и 28 столбцов
- test-набор содержит 966 строк и 27 столбцов
- в данных присутствуют числовые и категориальные признаки
- целевая переменная присутствует только в обучающей выборке

## Основные группы признаков

В датасете представлены:
- демографические признаки
- медицинские показатели
- показатели образа жизни
- биохимические показатели крови

Целевая переменная: `Heart Attack Risk (Binary)` — бинарный признак риска сердечного приступа.

In [5]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8685 entries, 0 to 8684
Data columns (total 28 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Unnamed: 0                       8685 non-null   int64  
 1   Age                              8685 non-null   float64
 2   Cholesterol                      8685 non-null   float64
 3   Heart rate                       8685 non-null   float64
 4   Diabetes                         8442 non-null   float64
 5   Family History                   8442 non-null   float64
 6   Smoking                          8442 non-null   float64
 7   Obesity                          8442 non-null   float64
 8   Alcohol Consumption              8442 non-null   float64
 9   Exercise Hours Per Week          8685 non-null   float64
 10  Diet                             8685 non-null   int64  
 11  Previous Heart Problems          8442 non-null   float64
 12  Medication Use      

## Анализ структуры данных

Анализ структуры показал, что в датасете присутствуют как числовые, так и категориальные признаки.

Также были обнаружены:
- пропущенные значения в части колонок
- категориальный признак `Gender`
- технические признаки, не несущие полезной информации для модели

In [7]:
TARGET = 'Heart Attack Risk (Binary)'

DROP_COLS = ['Unnamed: 0', 'id']

X = train.drop(columns=[TARGET] + DROP_COLS)
y = train[TARGET]

X_test = test.drop(columns=DROP_COLS)

print(X.shape)
print(y.shape)
print(X_test.shape)

(8685, 25)
(8685,)
(966, 25)


In [8]:
y.value_counts()

Heart Attack Risk (Binary)
0.0    5672
1.0    3013
Name: count, dtype: int64

In [9]:
X.isna().sum().sort_values(ascending=False)

Stress Level                       243
Diabetes                           243
Family History                     243
Smoking                            243
Obesity                            243
Alcohol Consumption                243
Previous Heart Problems            243
Medication Use                     243
Physical Activity Days Per Week    243
Triglycerides                        0
Systolic blood pressure              0
Gender                               0
Troponin                             0
CK-MB                                0
Blood sugar                          0
Sleep Hours Per Day                  0
Age                                  0
BMI                                  0
Income                               0
Sedentary Hours Per Day              0
Cholesterol                          0
Diet                                 0
Exercise Hours Per Week              0
Heart rate                           0
Diastolic blood pressure             0
dtype: int64

In [10]:
X = X.fillna(X.median(numeric_only=True))
X_test = X_test.fillna(X_test.median(numeric_only=True))

In [11]:
train['Gender'].value_counts()

Gender
Male      5882
Female    2560
1.0        156
0.0         87
Name: count, dtype: int64

In [12]:
train['Gender'].unique()

array(['Male', 'Female', '1.0', '0.0'], dtype=object)

In [13]:
train['Gender'] = train['Gender'].replace({
    '1.0': 'Male',
    '0.0': 'Female'
})

test['Gender'] = test['Gender'].replace({
    '1.0': 'Male',
    '0.0': 'Female'
})

In [14]:
train['Gender'].value_counts()

Gender
Male      6038
Female    2647
Name: count, dtype: int64

## Очистка категориальных признаков

В признаке `Gender` были обнаружены некорректные значения (`1.0` и `0.0` вместе с `Male` и `Female`).

Для корректной работы модели все значения были приведены к единому формату.

In [16]:
TARGET = 'Heart Attack Risk (Binary)'
DROP_COLS = ['Unnamed: 0', 'id']

X = train.drop(columns=[TARGET] + DROP_COLS)
y = train[TARGET]

X_test = test.drop(columns=DROP_COLS)

print(X.shape)
print(X_test.shape)

(8685, 25)
(966, 25)


## Отбор признаков

Из обучающей выборки были исключены признаки `id` и `Unnamed: 0`, так как они не содержат полезной информации для предсказания и могут вносить шум в модель.

In [18]:
X = X.fillna(X.median(numeric_only=True))
X_test = X_test.fillna(X_test.median(numeric_only=True))

## Обработка пропусков

Для числовых признаков пропущенные значения были заполнены медианными значениями. Такой подход позволяет сохранить объём выборки и уменьшить влияние выбросов.

In [20]:
cat_features = ['Gender']

## Категориальные признаки

В качестве категориального признака для модели был использован признак `Gender`.

In [22]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_valid.shape)

(6948, 25)
(1737, 25)


## Разделение выборки

Данные были разделены на обучающую и валидационную выборки в соотношении 80/20.

Для сохранения распределения целевой переменной использована стратификация.

In [24]:
model = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function='Logloss',
    eval_metric='AUC',
    random_seed=42,
    verbose=100
)

model.fit(
    X_train,
    y_train,
    cat_features=['Gender'],
    eval_set=(X_valid, y_valid),
    use_best_model=True
)

0:	test: 0.5213512	best: 0.5213512 (0)	total: 57.7ms	remaining: 28.8s
100:	test: 0.5507574	best: 0.5542262 (41)	total: 250ms	remaining: 988ms
200:	test: 0.5555029	best: 0.5576790 (192)	total: 441ms	remaining: 655ms
300:	test: 0.5680709	best: 0.5721729 (248)	total: 635ms	remaining: 420ms
400:	test: 0.5693724	best: 0.5721729 (248)	total: 829ms	remaining: 205ms
499:	test: 0.5672695	best: 0.5721729 (248)	total: 1.02s	remaining: 0us

bestTest = 0.572172939
bestIteration = 248

Shrink model to first 249 iterations.


CatBoostClassifier(depth=6, eval_metric='AUC', iterations=500, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=100)

## Обучение baseline-модели

В качестве baseline-модели выбран `CatBoostClassifier`, так как данный алгоритм хорошо подходит для табличных данных и позволяет удобно учитывать категориальные признаки.

In [26]:
y_pred = model.predict(X_valid)
y_proba = model.predict_proba(X_valid)[:, 1]

print("ROC-AUC:", roc_auc_score(y_valid, y_proba))
print("Accuracy:", accuracy_score(y_valid, y_pred))
print("F1:", f1_score(y_valid, y_pred))

ROC-AUC: 0.572172938950164
Accuracy: 0.6493955094991365
F1: 0.05581395348837209


In [27]:
leak_features = ['Troponin', 'CK-MB']

X_no_leak = X.drop(columns=leak_features)
X_test_no_leak = X_test.drop(columns=leak_features)

In [28]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X_no_leak,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [29]:
model2 = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function='Logloss',
    eval_metric='AUC',
    random_seed=42,
    verbose=100
)

model2.fit(
    X_train,
    y_train,
    cat_features=['Gender'],
    eval_set=(X_valid, y_valid),
    use_best_model=True
)

0:	test: 0.5292585	best: 0.5292585 (0)	total: 2.3ms	remaining: 1.15s
100:	test: 0.5475211	best: 0.5493447 (93)	total: 188ms	remaining: 745ms
200:	test: 0.5552075	best: 0.5557618 (129)	total: 375ms	remaining: 558ms
300:	test: 0.5562970	best: 0.5570662 (253)	total: 563ms	remaining: 372ms
400:	test: 0.5573192	best: 0.5629437 (345)	total: 747ms	remaining: 184ms
499:	test: 0.5628135	best: 0.5630724 (498)	total: 928ms	remaining: 0us

bestTest = 0.5630723514
bestIteration = 498

Shrink model to first 499 iterations.


CatBoostClassifier(depth=6, eval_metric='AUC', iterations=500, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=100)

## Результаты baseline-модели

Baseline-модель показала качество выше случайного уровня.

На валидационной выборке лучший результат был достигнут на 248 итерации, где значение ROC-AUC составило около 0.57.

In [31]:
y_pred = model2.predict(X_valid)
y_proba = model2.predict_proba(X_valid)[:, 1]

print("ROC-AUC:", roc_auc_score(y_valid, y_proba))
print("Accuracy:", accuracy_score(y_valid, y_pred))
print("F1:", f1_score(y_valid, y_pred))

ROC-AUC: 0.5630723513531695
Accuracy: 0.6465169833045481
F1: 0.14958448753462603


## Оценка качества модели

Модель была оценена на валидационной выборке с использованием нескольких метрик:

- ROC-AUC — показывает способность модели различать классы
- Accuracy — доля правильных ответов
- F1-score — баланс между precision и recall

Полученные результаты показывают, что модель способна выделять зависимости в данных, однако качество остаётся умеренным, что может быть связано со сложностью задачи и ограничениями данных.

In [33]:
feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model2.get_feature_importance()
})

feature_importance = feature_importance.sort_values(
    "importance",
    ascending=False
)

feature_importance.head(15)

,feature,importance
9,Diet,9.870471
15,BMI,7.777298
13,Sedentary Hours Per Day,7.741502
1,Cholesterol,7.563309
16,Triglycerides,7.218915
8,Exercise Hours Per Week,6.972122
14,Income,6.937006
21,Systolic blood pressure,6.778447
0,Age,6.551551
2,Heart rate,6.269272


## Анализ важности признаков

Для интерпретации модели был проведён анализ важности признаков.

Наиболее значимыми признаками оказались:
- Diet
- BMI
- Sedentary Hours Per Day
- Cholesterol
- Triglycerides
- Exercise Hours Per Week
- Артериальное давление
- Age

Полученные результаты выглядят логичными и соответствуют медицинской практике, что говорит о том, что модель выявляет осмысленные зависимости в данных.

In [35]:
thresholds = np.arange(0.1, 0.9, 0.01)

best_threshold = 0
best_f1 = 0

for t in thresholds:
    preds = (y_proba >= t).astype(int)
    f1 = f1_score(y_valid, preds)

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = t

print("Best threshold:", best_threshold)
print("Best F1:", best_f1)

Best threshold: 0.18999999999999995
Best F1: 0.527927927927928


## Подбор порога классификации

Стандартный порог 0.5 оказался не оптимальным для данной задачи.

Был выполнен подбор порога классификации, в результате чего удалось значительно повысить F1-score модели.

Это важно, так как F1-score лучше отражает качество модели при наличии дисбаланса классов.

In [37]:
model_final = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function='Logloss',
    eval_metric='AUC',
    random_seed=42,
    verbose=False
)

model_final.fit(
    X_no_leak,
    y,
    cat_features=['Gender']
)

CatBoostClassifier(depth=6, eval_metric='AUC', iterations=500, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=False)

In [38]:
test_proba = model_final.predict_proba(X_test_no_leak)[:, 1]

In [39]:
best_threshold = 0.19
test_pred = (test_proba >= best_threshold).astype(int)

In [40]:
submission = pd.DataFrame({
    'id': test['id'],
    'prediction': test_pred
})

submission.head()

,id,prediction
0,7746,1
1,4202,1
2,6632,1
3,4639,1
4,4825,1


In [41]:
submission.to_csv('../predictions/submission.csv', index=False)

In [42]:
os.listdir('../predictions')

['submission.csv']

In [43]:
joblib.dump(model_final, '../models/model_final.pkl')

['../models/model_final.pkl']

In [44]:
joblib.dump(best_threshold, '../models/best_threshold.pkl')

['../models/best_threshold.pkl']

In [45]:
joblib.dump(X_no_leak.columns.tolist(), '../models/feature_columns.pkl')

['../models/feature_columns.pkl']

In [46]:
os.listdir('../models')

['model_final.pkl', 'feature_columns.pkl', 'best_threshold.pkl']

## Итоговый вывод

В рамках проекта был реализован полный цикл разработки ML-решения:

- проведён анализ данных
- выполнена предобработка данных
- обучена модель CatBoost
- проведён анализ важности признаков
- выполнен подбор оптимального порога классификации
- подготовлены предсказания для тестовой выборки
- сохранены артефакты модели для дальнейшего использования
- реализован API-сервис на FastAPI

Модель показывает умеренное качество и может использоваться как базовое решение для оценки риска сердечного приступа.

Проект оформлен в виде полноценного сервиса и готов к дальнейшему развитию.